In [1]:
import sys
import os
from pathlib import Path

auth_klasor_yolu = r"C:\Users\BARIŞ\Desktop\SOFTWARE ENGİNEERİNG PROJECT - NETWORK-BASED LABORATORY EXAM SYSTEM"

if auth_klasor_yolu not in sys.path:
    sys.path.append(auth_klasor_yolu)

from instructor_auth import generate_instructor_token
import asyncio
import websockets
import json

async def send_command(action_type, payload_data):
    uri = "ws://localhost:8765"
    try:
        async with websockets.connect(uri) as ws:
            if action_type == "yeni_sinav":
                # 1. Sınav açmak için "admin" yetkisinde kriptografik token üretiyoruz
                admin_token = generate_instructor_token("admin_01", "admin")
                
                await ws.send(json.dumps({
                    "action": "register_exam",
                    "instructor_token": admin_token,  #  GÜVENLİK TOKEN'I
                    "payload": payload_data
                }))
                
                # Sunucudan gelen cevabı dinle (Hata varsa görmek için)
                response = json.loads(await ws.recv())
                if response.get("status") == "error":
                    print(f"❌ REDDEDİLDİ: {response.get('message')}")
                else:
                    print(f"✅ Sınav eklendi: {payload_data['exam_id']}")
                    
            elif action_type == "ogrenci_devam":
                # 2. Öğrenci affetmek için "teacher" yetkisinde kriptografik token üretiyoruz
                teacher_token = generate_instructor_token("hoca_01", "teacher")
                
                await ws.send(json.dumps({
                    "action": "resume_student",
                    "instructor_token": teacher_token,  #  GÜVENLİK TOKEN'I
                    "student_id": payload_data["student_id"]
                }))
                print(f"✅ Komut iletildi: {payload_data['student_id']} numaralı öğrenci affedildi ve devam ettiriliyor.")
                
    except Exception as e:
        print("❌ Sunucuya bağlanılamadı:", e)

# --- KULLANIM ALANI ---

# 1. Yeni Sınav Eklemek İstediğinde Başındaki '#' İşaretini Kaldır ve Çalıştır:
# await send_command("yeni_sinav", {"exam_id": "Applied Deep Learning", "total_duration_minutes": 15})

# 2. Duran Öğrenciyi Devam Ettirmek İstediğinde Başındaki '#' İşaretini Kaldır ve Çalıştır:
# Kendi test öğrencinin ID'sini yaz:
await send_command("ogrenci_devam", {"student_id": "std_01"})

✅ Komut iletildi: std_01 numaralı öğrenci affedildi ve devam ettiriliyor.
